# 01 - Exploratory Data Analysis

**Dataset:** MIMIC-CXR (Kaggle: `simhadrisadaram/mimic-cxr-dataset`)

This notebook inspects the raw dataset before building the pipeline:
- Confirms the report (`text`) column
- Report length distribution
- Missing values & duplicates
- Image paths & whether files exist on disk
- CheXpert-style label columns (if any)
- Tests section parsing (FINDINGS / IMPRESSION)

Run this **locally** - it is CPU-only and lightweight.


In [1]:
import sys
from pathlib import Path

# Make the repo root importable
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists() and (REPO_ROOT.parent / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import config
from src.utils import parse_report_sections

pd.set_option("display.max_colwidth", 200)
print("Repo root:   ", REPO_ROOT)
print("Raw data dir:", config.RAW_DATA_DIR)


ModuleNotFoundError: No module named 'config'

In [ ]:
# Auto-detect the dataset CSV inside data/raw/
csv_files = list(config.RAW_DATA_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  - {f.relative_to(config.RAW_DATA_DIR)}  ({f.stat().st_size/1e6:.1f} MB)")

assert csv_files, "No CSV in data/raw/. Run: python src/data/download.py"

# Load the largest CSV (most likely the main dataset)
main_csv = max(csv_files, key=lambda f: f.stat().st_size)
print("\nLoading:", main_csv.name)
df = pd.read_csv(main_csv)
print("Shape:", df.shape)


In [ ]:
print("Columns:")
for c in df.columns:
    print(f"  {c:30s} {str(df[c].dtype):12s} nunique={df[c].nunique()}")
df.head(3)


## Report text column

The assignment states *"the text column is the report"*. Confirm and inspect it.


In [ ]:
text_col_candidates = [c for c in df.columns
                       if c.lower() in ("text", "report", "findings", "impression", "report_text")]
TEXT_COL = text_col_candidates[0] if text_col_candidates else None
print("Report text column:", TEXT_COL)

if TEXT_COL:
    for i in range(3):
        print(f"\n--- Sample report {i} ---")
        print(df[TEXT_COL].iloc[i])


In [ ]:
if TEXT_COL:
    df["_report_str"] = df[TEXT_COL].fillna("").astype(str)
    df["_n_chars"] = df["_report_str"].str.len()
    df["_n_words"] = df["_report_str"].str.split().str.len()
    print(df[["_n_chars", "_n_words"]].describe())


In [ ]:
if TEXT_COL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["_n_words"].hist(bins=50, ax=axes[0])
    axes[0].set_title("Report length (words)")
    axes[0].set_xlabel("words")
    df["_n_chars"].hist(bins=50, ax=axes[1])
    axes[1].set_title("Report length (chars)")
    axes[1].set_xlabel("chars")
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / "eda_report_lengths.png", dpi=100)
    plt.show()


## Missing values & duplicates


In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
if TEXT_COL:
    n_empty = (df["_report_str"].str.strip() == "").sum()
    n_dup = df["_report_str"].duplicated().sum()
    print(f"Empty reports:     {n_empty}")
    print(f"Duplicate reports: {n_dup}")


## Image paths

Check whether the dataset references image files and whether they exist on disk.


In [ ]:
img_col_candidates = [c for c in df.columns
                      if any(k in c.lower() for k in ("image", "path", "dicom", "jpg", "png", "file"))]
print("Possible image-path columns:", img_col_candidates)

IMG_COL = img_col_candidates[0] if img_col_candidates else None


def resolve(p):
    p = str(p)
    for base in [Path(p), config.RAW_DATA_DIR / p, main_csv.parent / p]:
        if base.exists():
            return base
    return None


if IMG_COL:
    print("\nSample values:")
    for p in df[IMG_COL].dropna().astype(str).head(5):
        print("  ", p)
    resolved = [resolve(p) for p in df[IMG_COL].dropna().astype(str).head(50)]
    n_found = sum(r is not None for r in resolved)
    print(f"\nResolved {n_found}/50 sampled image paths on disk")


In [ ]:
# Display a sample X-ray if images are available
if IMG_COL:
    shown = False
    for p in df[IMG_COL].dropna().astype(str):
        rp = resolve(p)
        if rp is not None:
            img = Image.open(rp).convert("L")
            plt.figure(figsize=(5, 5))
            plt.imshow(img, cmap="gray")
            plt.title(f"Sample X-ray: {Path(p).name}")
            plt.axis("off")
            plt.show()
            print("Image size:", img.size)
            shown = True
            break
    if not shown:
        print("No image files could be resolved on disk yet.")


## CheXpert-style label columns

The starter repo drives QA generation from CheXpert labels (14 findings). Check
whether this dataset includes them - if so, `qa_builder.py` can use them to
balance question types.


In [ ]:
CHEXPERT_LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
present_labels = [c for c in df.columns if c in CHEXPERT_LABELS]
print(f"CheXpert label columns present: {len(present_labels)}/14")
print(present_labels)

if present_labels:
    print("\nLabel value counts (1=present, 0=absent, -1=uncertain):")
    for lbl in present_labels:
        print(f"\n{lbl}:")
        print(df[lbl].value_counts(dropna=False))


## Section parsing test

`src/utils.parse_report_sections` splits reports into FINDINGS / IMPRESSION.
Verify it behaves on this dataset's report format.


In [ ]:
if TEXT_COL:
    for i in range(3):
        sections = parse_report_sections(df["_report_str"].iloc[i])
        print(f"--- Report {i} ---")
        print("FINDINGS:  ", sections["findings"][:150])
        print("IMPRESSION:", sections["impression"][:150])
        print()


## Summary & takeaways

Use the printout below when writing `src/data/preprocess.py` and `qa_builder.py`.


In [ ]:
print("=" * 55)
print("EDA SUMMARY")
print("=" * 55)
print(f"Rows:                {len(df)}")
print(f"Columns:             {list(df.columns)}")
print(f"Report text column:  {TEXT_COL}")
print(f"Image path column:   {IMG_COL}")
print(f"CheXpert labels:     {len(present_labels)}/14 present")
if TEXT_COL:
    print(f"Median report words: {df['_n_words'].median():.0f}")
    print(f"Empty reports:       {(df['_report_str'].str.strip() == '').sum()}")
    print(f"Duplicate reports:   {df['_report_str'].duplicated().sum()}")
print("=" * 55)
